# Fake review detection: every preprocessing step inside a Pipeline

`notebook.ipynb` fitted the TF-IDF vectorizer on **all** reviews before splitting, so the vocabulary and IDF weights had already seen the test reviews. The behavioural features were also added unscaled next to TF-IDF values.

Here the text cleaning, TF-IDF, behavioural features, their scaling and the classifier are one scikit-learn `Pipeline`. Cross-validation and the grid search refit it on each training fold only, and the saved model takes raw review text.

Data: `data/fake reviews dataset.csv` (Salminen et al.): `CG` = computer-generated (fake), `OR` = original (real).

In [1]:
import sys
sys.path.insert(0, "../app")

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.svm import LinearSVC

from text_features import behavior_features, clean_texts, to_frame

df = pd.read_csv("../data/fake reviews dataset.csv")
X = df["text_"].astype(str)
y = (df["label"] == "OR").astype(int)  # 1 = real, 0 = fake
print(df.shape)
y.value_counts()

(40432, 4)


label
0    20216
1    20216
Name: count, dtype: int64

## Hold out a test set before anything is fitted

In [2]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
len(X_train), len(X_test)

(32345, 8087)

## The pipeline

Two branches read the same raw text:
- **words**: clean the text, then TF-IDF on unigrams and bigrams (top 5,000);
- **style**: length, word count, `!` count, capital ratio, repetition, mean word length, then scaling so they sit on the same scale as TF-IDF values.

In [3]:
features = ColumnTransformer([
    ("words", Pipeline([("clean", FunctionTransformer(clean_texts)),
                        ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2)))]), "text"),
    ("style", Pipeline([("extract", FunctionTransformer(behavior_features)),
                        ("scale", StandardScaler())]), "text"),
])
pipe = Pipeline([("frame", FunctionTransformer(to_frame)), ("features", features), ("svm", LinearSVC(C=0.1))])
scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1)
print(f"Pipeline CV F1 {scores.mean():.4f} ± {scores.std():.4f}")

Pipeline CV F1 0.8986 ± 0.0038


## For comparison: the old, leaky way

Same words-only model, with TF-IDF fitted on all training reviews before cross-validation (as in `notebook.ipynb`) versus inside the pipeline. On 32k reviews the IDF weights barely depend on any one fold, so the gap is tiny; the pipeline score is still the one to report, and the same habit causes large errors with feature selection, target encoding or oversampling.

In [4]:
tfidf_all = TfidfVectorizer(max_features=5000, ngram_range=(1, 2)).fit(clean_texts(X_train))
X_leaky = tfidf_all.transform(clean_texts(X_train))
leaky = cross_val_score(LinearSVC(C=0.1), X_leaky, y_train, cv=cv, scoring="f1", n_jobs=-1)

words_only = Pipeline([("clean", FunctionTransformer(clean_texts)),
                       ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
                       ("svm", LinearSVC(C=0.1))])
honest = cross_val_score(words_only, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1)
print(f"Words only, TF-IDF fitted on all rows (leaky): CV F1 {leaky.mean():.4f}")
print(f"Words only, TF-IDF inside the pipeline:        CV F1 {honest.mean():.4f}")
print(f"Words + style features, full pipeline:         CV F1 {scores.mean():.4f}")

Words only, TF-IDF fitted on all rows (leaky): CV F1 0.8807
Words only, TF-IDF inside the pipeline:        CV F1 0.8802
Words + style features, full pipeline:         CV F1 0.8986


## Tune C inside the pipeline

In [5]:
grid = GridSearchCV(pipe, {"svm__C": [0.01, 0.1, 1]}, cv=cv, scoring="f1", n_jobs=-1)
grid.fit(X_train, y_train)
print("best:", grid.best_params_, " CV F1:", round(grid.best_score_, 4))

best: {'svm__C': 1}  CV F1: 0.9018


## Final check on the untouched test set

In [6]:
best = grid.best_estimator_
pred = best.predict(X_test)
print(f"Test accuracy {accuracy_score(y_test, pred):.4f}   Test F1 {f1_score(y_test, pred):.4f}")
print(classification_report(y_test, pred, target_names=["fake (CG)", "real (OR)"]))

Test accuracy 0.9014   Test F1 0.9020
              precision    recall  f1-score   support

   fake (CG)       0.91      0.90      0.90      4044
   real (OR)       0.90      0.91      0.90      4043

    accuracy                           0.90      8087
   macro avg       0.90      0.90      0.90      8087
weighted avg       0.90      0.90      0.90      8087



## Save one artifact

It replaces `tfidf_vectorizer.pkl` + `linear_svm_model_adv.pkl` and takes raw text. The app needs `app/text_features.py` importable to load it.

In [7]:
joblib.dump(best, "../models/review_pipeline.joblib")
loaded = joblib.load("../models/review_pipeline.joblib")
samples = ["This product is amazing, high quality and worth the price",
           "BEST PRODUCT EVER!!! AMAZING AMAZING BUY NOW!!!",
           "The zipper broke after two weeks; customer service replaced it, but the stitching is still loose."]
["real" if p else "fake" for p in loaded.predict(samples)]

['real', 'fake', 'real']